# 01 — Ingestion: NHS England MHSDS Time Series

**Project:** ADHD Care Equity Tracker UK
**Notebook purpose:** Download and persist the NHS England Mental Health Services Data Set (MHSDS) monthly time-series CSV — the foundational dataset covering April 2016 to the latest published month of activity data for secondary mental health services in England.
**Author:** Noble Chidera Onyema
**Created:** 15 May 2026

---

© 2026 Noble Chidera Onyema. All Rights Reserved.
See `LICENSE` in the repository root. No commercial use, derivative works, redistribution, or use as ML training data without written permission.

In [1]:
"""
01_ingestion.ipynb — NHS England MHSDS time-series download.

Copyright (c) 2026 Noble Chidera Onyema. All Rights Reserved.
"""

from pathlib import Path
import sys
import requests
import pandas as pd

# Resolve paths relative to the notebook location so the code works
# whether run interactively or imported elsewhere in the project.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Confirm the folders we scaffolded on Day 1 actually exist where we expect them.
assert DATA_RAW.exists(), f"data/raw not found at {DATA_RAW}"
assert DATA_PROCESSED.exists(), f"data/processed not found at {DATA_PROCESSED}"

print(f"Python:          {sys.version.split()[0]}")
print(f"pandas:          {pd.__version__}")
print(f"requests:        {requests.__version__}")
print(f"Project root:    {PROJECT_ROOT}")
print(f"data/raw:        {DATA_RAW}")
print(f"data/processed:  {DATA_PROCESSED}")

Python:          3.11.9
pandas:          2.2.3
requests:        2.32.3
Project root:    C:\Users\HP\Projects\adhd-care-equity-tracker
data/raw:        C:\Users\HP\Projects\adhd-care-equity-tracker\data\raw
data/processed:  C:\Users\HP\Projects\adhd-care-equity-tracker\data\processed


## Step 1 — Download the MHSDS time-series archive

We download the official NHS England Mental Health Services Data Set (MHSDS) monthly time-series file, which covers **April 2016 to the latest published performance month** (February 2026 as of this notebook's creation). The file is distributed as a ZIP archive containing a CSV with selected MHSDS measures aggregated by month and demographic / organisational breakdown.

**Source:** NHS England Digital — Mental Health Services Monthly Statistics
**Source page:** [digital.nhs.uk/.../mental-health-services-monthly-statistics/performance-february-2026](https://digital.nhs.uk/data-and-information/publications/statistical/mental-health-services-monthly-statistics/performance-february-2026)
**Download URL:** `files.digital.nhs.uk` (verified live, 15 May 2026)
**Licence of source data:** Open Government Licence v3.0 — freely usable with attribution. Our analysis and code remain under the All Rights Reserved licence of this repository.

The download cell is idempotent: re-running it does not re-download if the file is already present in `data/raw/`. The raw archive itself is git-ignored — only our analytical work is committed.

In [3]:
import zipfile

# NHS England MHSDS Time Series — verified live 15 May 2026
MHSDS_URL = "https://files.digital.nhs.uk/18/45D7A4/MHSDS%20Time_Series_data_Apr_2016_Feb_Perf_2026.zip"

# Save under a clean snake_case filename rather than the URL-encoded original.
ZIP_PATH = DATA_RAW / "mhsds_time_series_apr2016_feb2026.zip"

if ZIP_PATH.exists():
    size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
    print(f"Already downloaded: {ZIP_PATH.name} ({size_mb:.1f} MB) — skipping fetch.")
else:
    # NHS Digital's CDN is sensitive about clients without a real User-Agent header.
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }

    print(f"Downloading {MHSDS_URL} ...")
    with requests.get(MHSDS_URL, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        reported = int(response.headers.get("Content-Length", 0))
        print(f"Reported size: {reported / 1024 / 1024:.1f} MB")

        written = 0
        with open(ZIP_PATH, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 64):
                f.write(chunk)
                written += len(chunk)

    print(f"\nWrote: {ZIP_PATH}")
    print(f"Final size: {written / 1024 / 1024:.1f} MB")

Reported size: 26.1 MB

Wrote: C:\Users\HP\Projects\adhd-care-equity-tracker\data\raw\mhsds_time_series_apr2016_feb2026.zip
Final size: 26.1 MB


In [4]:
# Peek inside the ZIP before unpacking — we want to know what we're getting.
with zipfile.ZipFile(ZIP_PATH) as zf:
    members = zf.infolist()
    print(f"Archive contains {len(members)} file(s):\n")
    for m in members:
        size_mb = m.file_size / 1024 / 1024
        print(f"  {m.filename:65s} {size_mb:8.2f} MB")

Archive contains 5 file(s):

  MHSDS Time_Series_data_Apr_2016_Feb_Perf_2026/                        0.00 MB
  MHSDS Time_Series_data_Apr_2016_Feb_Perf_2026/MHSDS Time_Series_data_Apr_2016_MarPrf_2023.csv    89.93 MB
  MHSDS Time_Series_data_Apr_2016_Feb_Perf_2026/MHSDS Time_Series_data_Apr_2023_MarFinal_2024.csv   155.76 MB
  MHSDS Time_Series_data_Apr_2016_Feb_Perf_2026/MHSDS Time_Series_data_Apr_2024_Mar_2025_Final.csv   178.94 MB
  MHSDS Time_Series_data_Apr_2016_Feb_Perf_2026/MHSDS Time_Series_data_Feb_Perf_2026.csv   166.90 MB


## Step 2 — Inspect archive contents and extract

The downloaded archive contains **four CSVs** covering different time slices of the MHSDS time-series. NHS England publishes the historical record in revised "Final" annual blocks (more accurate, updated retrospectively) and the latest month as a "Performance" snapshot. The contents are:

| File | Period | Type | Approx. size |
|---|---|---|---|
| `…_Apr_2016_MarPrf_2023.csv` | Apr 2016 → Mar 2023 | Performance | 90 MB |
| `…_Apr_2023_MarFinal_2024.csv` | Apr 2023 → Mar 2024 | Final | 156 MB |
| `…_Apr_2024_Mar_2025_Final.csv` | Apr 2024 → Mar 2025 | Final | 179 MB |
| `…_Feb_Perf_2026.csv` | Feb 2026 (single month) | Performance | 167 MB |

**Known gap:** April 2025 → January 2026 (10 months) is not in this archive — those increments live in the previous monthly publications (Mar 2025, Apr 2025, May 2025, …, Jan 2026 releases). We document this limitation in the README and may backfill in a later iteration. For Week 1 EDA, ~9 years of historical data is more than enough to characterise regional inequity.

We extract the archive into `data/raw/`, then peek at one CSV's schema before loading anything large into memory.

In [5]:
EXTRACT_DIR = DATA_RAW / "mhsds_time_series_apr2016_feb2026"

if EXTRACT_DIR.exists() and any(EXTRACT_DIR.rglob("*.csv")):
    print(f"Already extracted to: {EXTRACT_DIR} — skipping.")
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {ZIP_PATH.name} ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to: {EXTRACT_DIR}")

# List every CSV that ended up on disk.
csv_paths = sorted(EXTRACT_DIR.rglob("*.csv"))
print(f"\n{len(csv_paths)} CSV file(s) found:")
for p in csv_paths:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:65s} {size_mb:8.2f} MB")

Extracting mhsds_time_series_apr2016_feb2026.zip ...
Extracted to: C:\Users\HP\Projects\adhd-care-equity-tracker\data\raw\mhsds_time_series_apr2016_feb2026

4 CSV file(s) found:
  MHSDS Time_Series_data_Apr_2016_MarPrf_2023.csv                      89.93 MB
  MHSDS Time_Series_data_Apr_2023_MarFinal_2024.csv                   155.76 MB
  MHSDS Time_Series_data_Apr_2024_Mar_2025_Final.csv                  178.94 MB
  MHSDS Time_Series_data_Feb_Perf_2026.csv                            166.90 MB


In [7]:
# Peek at the most recent file's schema using only the first 5 rows.
# This is fast — it does not load the 167 MB file into memory.
sample_path = next(p for p in csv_paths if "Feb_Perf_2026" in p.name)
print(f"Sampling: {sample_path.name}\n")

sample = pd.read_csv(sample_path, nrows=5, low_memory=False)
print(f"Shape of sample (5 rows): {sample.shape}")
print(f"Columns ({len(sample.columns)}):")
for col in sample.columns:
    print(f"  - {col}")

print("\nFirst 5 rows:")
sample.head()

Sampling: MHSDS Time_Series_data_Feb_Perf_2026.csv

Shape of sample (5 rows): (5, 11)
Columns (11):
  - REPORTING_PERIOD_START
  - REPORTING_PERIOD_END
  - STATUS
  - BREAKDOWN
  - PRIMARY_LEVEL
  - PRIMARY_LEVEL_DESCRIPTION
  - SECONDARY_LEVEL
  - SECONDARY_LEVEL_DESCRIPTION
  - MEASURE_ID
  - MEASURE_NAME
  - MEASURE_VALUE

First 5 rows:


,REPORTING_PERIOD_START,REPORTING_PERIOD_END,STATUS,BREAKDOWN,PRIMARY_LEVEL,PRIMARY_LEVEL_DESCRIPTION,SECONDARY_LEVEL,SECONDARY_LEVEL_DESCRIPTION,MEASURE_ID,MEASURE_NAME,MEASURE_VALUE
0,01/04/2025,30/04/2025,Performance,Sub ICB - GP Practice or Residence,00L,NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L,NONE,NONE,MHS24a,Under 16 bed days on adult wards in reporting ...,*
1,01/04/2025,30/04/2025,Performance,Sub ICB - GP Practice or Residence,00L,NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L,NONE,NONE,MHS24b,Age 16 bed days on adult wards in reporting pe...,*
2,01/04/2025,30/04/2025,Performance,Sub ICB - GP Practice or Residence,00L,NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L,NONE,NONE,MHS24c,Age 17 bed days on adult wards in reporting pe...,*
3,01/04/2025,30/04/2025,Performance,Sub ICB - GP Practice or Residence,00L,NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L,NONE,NONE,MHS29,Contacts in RP,15330
4,01/04/2025,30/04/2025,Performance,Sub ICB - GP Practice or Residence,00L,NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L,NONE,NONE,MHS69,"The number of children and young people, regar...",705


## Step 3 — Load and concatenate all four CSVs

We load each of the four time-series CSVs as a pandas DataFrame, then concatenate them into a single tidy long-format dataset spanning **April 2016 → February 2026** (with the documented gap of April 2025 → January 2026).

Combined uncompressed size is roughly 591 MB across the four files, which is well within memory for a modern laptop. We use `low_memory=False` to let pandas scan each column fully and pick the right dtype on the first pass — slower than streaming, but it produces cleaner dtypes than the chunked alternative.

In [8]:
# Load every CSV in deterministic order and concatenate.
frames = []
for path in csv_paths:
    print(f"Loading {path.name} ...")
    df = pd.read_csv(path, low_memory=False)
    df["SOURCE_FILE"] = path.name  # provenance — which file each row came from
    print(f"  shape: {df.shape}")
    frames.append(df)

mhsds = pd.concat(frames, ignore_index=True)
print(f"\nCombined dataframe shape: {mhsds.shape}")
print(f"Memory footprint: {mhsds.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

# Parse dates so they're proper datetime, not strings.
mhsds["REPORTING_PERIOD_START"] = pd.to_datetime(
    mhsds["REPORTING_PERIOD_START"], dayfirst=True, errors="coerce"
)
mhsds["REPORTING_PERIOD_END"] = pd.to_datetime(
    mhsds["REPORTING_PERIOD_END"], dayfirst=True, errors="coerce"
)

# Quick sanity checks — what time range, what statuses, how many unique measures.
print(f"\nReporting period range: {mhsds['REPORTING_PERIOD_START'].min().date()}  →  {mhsds['REPORTING_PERIOD_END'].max().date()}")
print(f"Distinct STATUS values: {sorted(mhsds['STATUS'].dropna().unique().tolist())}")
print(f"Distinct MEASURE_ID count: {mhsds['MEASURE_ID'].nunique()}")
print(f"Distinct BREAKDOWN count: {mhsds['BREAKDOWN'].nunique()}")
print(f"Rows missing MEASURE_VALUE: {mhsds['MEASURE_VALUE'].isna().sum()}")

Loading MHSDS Time_Series_data_Apr_2016_MarPrf_2023.csv ...
  shape: (510315, 12)
Loading MHSDS Time_Series_data_Apr_2023_MarFinal_2024.csv ...
  shape: (730350, 12)
Loading MHSDS Time_Series_data_Apr_2024_Mar_2025_Final.csv ...
  shape: (829055, 12)
Loading MHSDS Time_Series_data_Feb_Perf_2026.csv ...
  shape: (751416, 12)

Combined dataframe shape: (2821136, 12)
Memory footprint: 2522.4 MB

Reporting period range: 2016-01-04  →  2026-02-28
Distinct STATUS values: ['Final', 'Performance', 'Performance ']
Distinct MEASURE_ID count: 121
Distinct BREAKDOWN count: 27
Rows missing MEASURE_VALUE: 0


## Data notes

- `STATUS` has a duplicate caused by trailing whitespace: `'Performance'` vs `'Performance '`. Strip before saving.
- Frame sits at 2.5 GB in memory. Most string columns repeat heavily. Cast to `category` before writing parquet.
- Coverage gap Apr 2025 to Jan 2026, as expected from the archive contents.

In [9]:
for col in mhsds.select_dtypes(include="object").columns:
    mhsds[col] = mhsds[col].str.strip()

cat_cols = [
    "STATUS", "BREAKDOWN",
    "PRIMARY_LEVEL", "PRIMARY_LEVEL_DESCRIPTION",
    "SECONDARY_LEVEL", "SECONDARY_LEVEL_DESCRIPTION",
    "MEASURE_ID", "MEASURE_NAME",
    "SOURCE_FILE",
]
for col in cat_cols:
    mhsds[col] = mhsds[col].astype("category")

print(f"Memory: {mhsds.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
print(f"STATUS: {sorted(mhsds['STATUS'].dropna().unique().tolist())}")

parquet_path = DATA_PROCESSED / "mhsds_time_series_2016_2026.parquet"
mhsds.to_parquet(parquet_path, engine="pyarrow", compression="snappy", index=False)
print(f"Saved {parquet_path.name} ({parquet_path.stat().st_size / 1024 / 1024:.1f} MB)")

Memory: 235.6 MB
STATUS: ['Final', 'Performance']
Saved mhsds_time_series_2016_2026.parquet (4.4 MB)


## AWT reference tables

Small zip on the same NHS publication page. Single month (Feb 2026). Explicit waiting-time metrics rather than general activity counts.

In [10]:
AWT_URL = "https://files.digital.nhs.uk/BF/64B055/MHSDS%20AWT_FebPerf_2026.zip"
AWT_ZIP = DATA_RAW / "mhsds_awt_feb2026.zip"

if not AWT_ZIP.exists():
    with requests.get(AWT_URL, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(AWT_ZIP, "wb") as f:
            for chunk in r.iter_content(chunk_size=64 * 1024):
                f.write(chunk)

print(f"{AWT_ZIP.name}: {AWT_ZIP.stat().st_size / 1024:.1f} KB")

with zipfile.ZipFile(AWT_ZIP) as zf:
    for m in zf.infolist():
        print(f"  {m.filename}  {m.file_size / 1024:.1f} KB")

mhsds_awt_feb2026.zip: 160.2 KB
  MHSDS AWT_FebPerf_2026/  0.0 KB
  MHSDS AWT_FebPerf_2026/MHSDS AWT_FebPerf_2026.xlsx  223.7 KB
